# TP 1.2 - LLM as a Judge

Ce notebook introduit le concept de **LLM as a Judge** : utiliser un modèle de langage pour évaluer les sorties d'un autre modèle.

Déroulement en 3 étapes :
1. Un LLM **génère** 20 candidats pour répondre à la requête utilisateur
2. Un LLM **juge** et note chaque candidat de 1 à 10
3. Le code **sélectionne** les 5 candidats avec les meilleures notes

### 0.1. Documentation générale des librairies utilisées

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

In [ ]:
import json

from pydantic import BaseModel

from shared.llm_utils import (
    LLMRequest,
    run_llm_structured, # Cloud
    #run_llm_local_structured as run_llm_structured, # Local
)

### 0.2. Séparer prompt système et requête utilisateur

**Attention à bien séparer le prompt système (objectif général du LLM/Agent) et la requête utilisateur (besoin spécifique).**

Ici, par exemple :
- La requête utilisateur demande seulement des lieux à visiter et les contraintes (petit budget).
- Le prompt système du **LLM générateur** lui indique qu’il doit générer 20 réponses (sans parler de l'use case spécifique, de la ville ou des contraintes).
- Le prompt système du **LLM juge** lui explique son objectif : noter chaque proposition en fonction du besoin utilisateur (sans avoir à préciser tout le contexte).

---
## 1. Générer 20 candidats

On demande au LLM de générer 20 candidats pour répondre à la requête utilisateur, en JSON structuré (nom, description, coût estimé).

Le prompt système doit être **générique** : il ne doit pas mentionner le contexte spécifique (ville, budget, contraintes) — c'est la requête utilisateur qui porte ces informations.

Comme en TP1_1, on utilise `run_llm_structured` (sortie structurée) plutôt que de décrire le JSON en texte.

### 1.1. Définir la structure de sortie et le system prompt

In [ ]:
user_query = "Je visite Paris avec un petit budget. Propose-moi des lieux à visiter."

# DOC (response_schema) : https://ai.google.dev/gemini-api/docs/structured-output
class PlaceItem(BaseModel):
    name: str
    description: str
    estimated_cost_eur: float


class GeneratedPlaces(BaseModel):
    places: list[PlaceItem]


system_prompt_generator = """
Tu es un assistant expert qui génère des propositions détaillées pour répondre aux requêtes utilisateur.
Génère exactement 20 candidats adaptés à la demande.

Contraintes :
- Varie les types de propositions pour couvrir différentes options.
- Adapte chaque proposition au contexte de la requête utilisateur.

Pour chaque candidat, fournis :
- un nom court
- une description en 1 phrase
- un coût estimé en euros (0 si gratuit)
"""

### 1.2. Appeler la fonction de sortie structurée

In [ ]:
request_generator = LLMRequest(system_prompt=system_prompt_generator, user_prompt=user_query)
result_generator = await run_llm_structured(request_generator, response_schema=GeneratedPlaces)

places_data = json.loads(result_generator.output)
places = places_data["places"]

print(f"{len(places)} lieux générés\n")
for i, place in enumerate(places, start=1):
    print(f"{i:2}. {place['name']} ({place['estimated_cost_eur']} EUR) - {place['description']}")

---
## 2. LLM as a Judge : noter chaque candidat

On passe les 20 candidats à un second appel LLM qui joue le rôle de **juge**.
Pour chaque candidat, le juge attribue une note de 1 à 10 et justifie son choix.

Le prompt système du juge doit être **générique** : il définit le rôle d'évaluateur sans reprendre les contraintes spécifiques de la requête. C'est le prompt utilisateur (avec la liste des candidats) qui fournit le contexte d'évaluation.

### 2.1. Définir la structure de sortie et le system prompt du juge

In [ ]:
class RatingItem(BaseModel):
    name: str
    score: int
    reason: str


class JudgeRatings(BaseModel):
    ratings: list[RatingItem]


system_prompt_judge = """
Tu es un juge impartial qui évalue des propositions selon leur pertinence et qualité.
On te fournit une liste de candidats avec leur description et coût estimé.
Tu dois noter chaque candidat de 1 à 10 et justifier ton choix en fonction de la demande utilisateur.
"""

### 2.2. Construire le prompt utilisateur et appeler la fonction de sortie structurée

In [ ]:
places_as_text = json.dumps(places, ensure_ascii=False, indent=2)
judge_user_prompt = f"Voici les lieux à évaluer :\n{places_as_text}"

request_judge = LLMRequest(system_prompt=system_prompt_judge, user_prompt=judge_user_prompt)
result_judge = await run_llm_structured(request_judge, response_schema=JudgeRatings)

ratings_data = json.loads(result_judge.output)
ratings = ratings_data["ratings"]

print(f"{len(ratings)} lieux évalués\n")
for rating in ratings:
    print(f"{rating['score']}/10 - {rating['name']} : {rating['reason']}")

---
## 3. Sélectionner le Top 5

On trie les candidats par note décroissante et on affiche les 5 meilleurs.

In [ ]:
ratings_sorted = sorted(ratings, key=lambda r: r["score"], reverse=True)
top_5 = ratings_sorted[:5]

print("TOP 5 - Meilleures recommandations")
print("=" * 60)
for rank, rating in enumerate(top_5, start=1):
    print(f"\n{rank}. {rating['name']} - Note : {rating['score']}/10")
    print(f"   {rating['reason']}")